# Querying Local Pipeline Data

This notebook shows how to query the pipeline's Delta tables stored on the
local filesystem — no S3 credentials needed. Use this when you've run the
pipeline locally with `python -m pipeline.run full`.

## How It Works

When `S3_BUCKET` is not set, the pipeline stores data under `<project>/data/`
in a medallion architecture:

```
data/
  raw/                          # Raw broker responses (encrypted values)
    ibkr_snapshot/
    trading212_snapshot/
    trading212_cdc/
    xtb_snapshot/
  normalized/                   # Cleaned, typed, joined positions
    ibkr_snapshot/
    trading212_snapshot/
    xtb_snapshot/
    consolidated_holdings/
  analytics/                    # Final output tables
    portfolio_allocation/
```

## Table Aliases

The query module registers Delta tables by short aliases:

| Alias | Layer | Description |
|-------|-------|-------------|
| `ibkr_snapshot_raw` | Raw | IBKR Flex raw data (encrypted) |
| `ibkr_snapshot_normalized` | Normalized | IBKR positions with decrypted metadata |
| `trading212_snapshot_raw` | Raw | T212 raw data (encrypted) |
| `trading212_snapshot_normalized` | Normalized | T212 positions |
| `xtb_snapshot_raw` | Raw | XTB raw data |
| `xtb_snapshot_normalized` | Normalized | XTB positions |
| `consolidated_holdings` | Normalized | Cross-broker merged holdings |
| `portfolio_allocation_analytics` | Analytics | Allocation by ticker

## 1. Connect to Local Data

No environment variables needed — the pipeline automatically uses local
storage when `S3_BUCKET` is not set.

In [ ]:
import os

# Make sure S3_BUCKET is not set — we want local storage
os.environ.pop("S3_BUCKET", None)

# (Optional) Set a custom data directory if you moved it
# os.environ["PIPELINE_DATA_DIR"] = "/path/to/data"

# Encryption key — needed to decrypt financial values
# Option 1: Set via env var (recommended)
# os.environ["ENCRYPTION_KEY"] = "your-fernet-key-here"
# Option 2: The pipeline will look for .secrets/encryption.key automatically

import polars as pl
from pipeline.query import list_tables, refresh, get_connection

pl.Config.set_tbl_rows(50)

In [ ]:
# Discover available Delta tables
refresh()
tables = list_tables()
print(f"Found {len(tables)} tables:")
for t in tables:
    print(f"  • {t}")

In [ ]:
db = get_connection()

## 2. Portfolio Allocation

The `portfolio_allocation_analytics` table is the final pipeline output —
percentage allocation per ticker, broken down by broker and currency.

In [ ]:
allocation = db.sql("SELECT * FROM portfolio_allocation_analytics").pl()
allocation

## 3. Currency Exposure

Aggregate by security currency to understand your FX exposure.

In [ ]:
db.sql("""
    SELECT
        security_currency,
        ROUND(SUM(percentage), 2) AS total_pct,
        COUNT(*) AS num_positions
    FROM portfolio_allocation_analytics
    GROUP BY security_currency
    ORDER BY total_pct DESC
""").pl()

## 4. Broker Distribution

How is your portfolio distributed across brokers?

In [ ]:
db.sql("""
    SELECT
        broker,
        ROUND(SUM(percentage), 2) AS total_pct,
        COUNT(*) AS num_positions
    FROM portfolio_allocation_analytics
    GROUP BY broker
    ORDER BY total_pct DESC
""").pl()

## 5. Top Holdings

Your largest positions by allocation percentage.

In [ ]:
db.sql("""
    SELECT ticker, description, security_currency, ROUND(percentage, 2) AS pct
    FROM portfolio_allocation_analytics
    ORDER BY percentage DESC
    LIMIT 10
""").pl()

## 6. Decrypt Raw Data

Raw tables store financial values as Fernet-encrypted bytes (type `binary`).
Use `decrypt_df` to automatically detect and decrypt all encrypted columns.

In [ ]:
from pipeline.query import decrypt_df

pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_width_chars(180)

In [ ]:
# Read and decrypt IBKR raw snapshot
raw_ibkr = db.sql("SELECT * FROM ibkr_snapshot_raw").pl()
decrypt_df(raw_ibkr).head(10)

## 7. Consolidated Holdings

The `consolidated_holdings` table merges positions from all brokers
into a single view. Values are encrypted — decrypt to see amounts.

In [ ]:
holdings = db.sql("""
    SELECT ticker, broker, security_currency, value, value_currency
    FROM consolidated_holdings
    ORDER BY value DESC
""").pl()
decrypt_df(holdings)

## 8. Cross-Broker Positions

Find tickers held in multiple brokers — useful for verifying deduplication.

In [ ]:
db.sql("""
    SELECT
        ticker,
        COUNT(DISTINCT broker) AS broker_count,
        ROUND(SUM(percentage), 2) AS total_pct
    FROM portfolio_allocation_analytics
    GROUP BY ticker
    HAVING COUNT(DISTINCT broker) > 1
    ORDER BY total_pct DESC
""").pl()

## 9. Per-Broker Normalized Data

Inspect the normalized data from each broker individually.

In [ ]:
# IBKR normalized snapshot
db.sql("""
    SELECT account_id, position_type, label, currency, isin, security_currency
    FROM ibkr_snapshot_normalized
    ORDER BY label
""").pl()

In [ ]:
# Trading 212 normalized snapshot
db.sql("""
    SELECT account_id, position_type, label, currency, isin, security_currency
    FROM trading212_snapshot_normalized
    ORDER BY label
""").pl()

## 10. Using Demo Mode Locally

To work with demo data locally, set `DEMO=true` before connecting:

```python
os.environ["DEMO"] = "true"
# Data will be read from data_demo/ instead of data/
# Or set a custom demo directory:
# os.environ["PIPELINE_DATA_DIR_DEMO"] = "/path/to/demo/data"
```

Demo mode writes to a separate `data_demo/` directory so your production
data stays isolated. All secrets use `_DEMO` variants (e.g., `ENCRYPTION_KEY_DEMO`).